# UrbanPulse AI

# Natural Language Processing (NLP)

## Objective

Build an AI system that understands complaint text, extracts useful information, predicts complaint category, and generates intelligent summaries for municipal authorities.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../data/complaints/bbmp_2024.csv")

df.head()

,Complaint ID,Category,Sub Category,Grievance Date,Ward Name,Grievance Status,Staff Remarks,Staff Name
0,20644646,Solid Waste (Garbage) Related,Garbage vehicle not arrived,2024-12-31 05:52:00.000000000,Manorayanapalya,Closed,Clear,Nagendra/JHI
1,20644640,Electrical,Street Light Not Working,2024-12-31 11:36:00.000000000,Basavanapura,Closed,Attended,Sachin Malagi/AE
2,20644639,Electrical,Street Light Not Working,2024-12-31 11:24:00.000000000,Maruthi Seva Nagar,Closed,Attended,Durga Prasad/AEE
3,20644638,Electrical,Street Light Not Working,2024-12-31 11:21:00.000000000,Aramane Nagar,Closed,Complaints Cleared,Suresh/AEE
4,20644637,Electrical,Street Light Not Working,2024-12-31 11:00:00.000000000,Vidyaranyapura,Closed,Resolved,Yelahanka AEE Electrical/AEE


In [3]:
df.columns = (
    df.columns
      .str.lower()
      .str.strip()
      .str.replace(" ","_")
)

In [4]:
df.columns

Index(['complaint_id', 'category', 'sub_category', 'grievance_date',
       'ward_name', 'grievance_status', 'staff_remarks', 'staff_name'],
      dtype='object')

# Text Preprocessing

## Objective

Clean complaint text before applying NLP techniques.

### Steps
- Convert text to lowercase
- Remove punctuation
- Remove numbers
- Remove extra spaces

In [5]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_text"] = df["staff_remarks"].apply(clean_text)

df[["staff_remarks", "clean_text"]].head()

,staff_remarks,clean_text
0,Clear,clear
1,Attended,attended
2,Attended,attended
3,Complaints Cleared,complaints cleared
4,Resolved,resolved


## Observation

- Converted text to lowercase.
- Removed numbers and special characters.
- Created a new column named `clean_text`.

# Tokenization

## Objective

Split text into individual words (tokens) for NLP processing.

In [8]:
!pip install nltk

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 2.1 MB/s eta 0:00:01
   -------------------- ------------------- 0.8/1.6 MB 2.0 MB/s eta 0:00:01
   --------------------------- ------------ 1.0/1.6 MB 1.5 MB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 1.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ayush\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [12]:
from nltk.tokenize import word_tokenize

print(word_tokenize("Street light is not working"))

['Street', 'light', 'is', 'not', 'working']


In [13]:
from nltk.tokenize import word_tokenize

df["tokens"] = df["clean_text"].apply(word_tokenize)

df[["clean_text", "tokens"]].head()

,clean_text,tokens
0,clear,[clear]
1,attended,[attended]
2,attended,[attended]
3,complaints cleared,"[complaints, cleared]"
4,resolved,[resolved]


# Stopword Removal

## Objective

Remove common English words that do not add meaningful information.

In [14]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

df["filtered_tokens"] = df["tokens"].apply(
    lambda words: [word for word in words if word not in stop_words]
)

df[["tokens", "filtered_tokens"]].head()

,tokens,filtered_tokens
0,[clear],[clear]
1,[attended],[attended]
2,[attended],[attended]
3,"[complaints, cleared]","[complaints, cleared]"
4,[resolved],[resolved]


## Observation

- Removed common words such as "is", "the", and "and".
- Remaining words contain more useful information for analysis.

# Lemmatization

## Objective

Convert words to their root form while preserving their meaning.

In [15]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

df["lemmatized_tokens"] = df["filtered_tokens"].apply(
    lambda words: [lemmatizer.lemmatize(word) for word in words]
)

df[["filtered_tokens", "lemmatized_tokens"]].head()

,filtered_tokens,lemmatized_tokens
0,[clear],[clear]
1,[attended],[attended]
2,[attended],[attended]
3,"[complaints, cleared]","[complaint, cleared]"
4,[resolved],[resolved]


## Observation

- Converted words into their base form.
- Reduced different word variations into a single root word.
- Prepared clean text for machine learning.

# Reconstruct Text

## Objective

Convert processed tokens back into text for feature extraction.

In [16]:
df["final_text"] = df["lemmatized_tokens"].apply(lambda x: " ".join(x))

df[["clean_text", "final_text"]].head()

,clean_text,final_text
0,clear,clear
1,attended,attended
2,attended,attended
3,complaints cleared,complaint cleared
4,resolved,resolved


## Observation

- Tokens are combined into a sentence.
- The processed text is ready for TF-IDF vectorization.

# TF-IDF Vectorization

## Objective

Convert processed text into numerical features that can be used for machine learning.

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X = tfidf.fit_transform(df["final_text"])

print(X.shape)

(207016, 5374)


## Observation

- Text data has been converted into numerical vectors.
- Each row represents one complaint.
- These vectors will be used to train the machine learning model.

In [19]:
feature_names = tfidf.get_feature_names_out()

print(feature_names[:20])

['aadhar' 'aasthi' 'aattended' 'aav' 'aaz' 'aazav' 'aazs' 'abc' 'ability'
 'abiove' 'abive' 'able' 'abnormally' 'absent' 'absolutely' 'abt' 'abuse'
 'acacia' 'acc' 'access']


In [20]:

print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 315288 stored elements and shape (207016, 5374)>
  Coords	Values
  (0, 910)	1.0
  (1, 358)	1.0
  (2, 358)	1.0
  (3, 1072)	0.68138284402621
  (3, 916)	0.7319271957420039
  (4, 4011)	1.0
  (5, 963)	1.0
  (6, 358)	1.0
  (7, 358)	1.0
  (8, 916)	1.0
  (9, 4011)	1.0
  (10, 4011)	1.0
  (11, 358)	1.0
  (12, 358)	1.0
  (13, 342)	1.0
  (14, 1072)	0.68138284402621
  (14, 916)	0.7319271957420039
  (15, 358)	1.0
  (16, 4011)	1.0
  (17, 358)	1.0
  (18, 1072)	0.68138284402621
  (18, 916)	0.7319271957420039
  (19, 358)	1.0
  (20, 4011)	1.0
  (21, 963)	1.0
  :	:
  (206997, 358)	0.32457785337537176
  (206997, 1095)	0.9458589837276145
  (206998, 1072)	0.17884598898641682
  (206998, 552)	0.3751676638999937
  (206998, 1168)	0.31748444180720553
  (206998, 524)	0.8523303147240857
  (206999, 358)	1.0
  (207000, 37)	0.6633065782892015
  (207000, 2381)	0.7483477688870805
  (207001, 358)	1.0
  (207002, 358)	0.5026699387158063
  (207002, 1072)	0.864478